# HunyuanVideo 1.5 — Colab T4

**Fast text-to-video generation** with HunyuanVideo 1.5 (Tencent) on a free T4 GPU.

### Why HunyuanVideo?
| | HunyuanVideo 1.5 | Wan 2.2 14B |
|---|---|---|
| **Model size** | ~7.76 GB (FP8) | ~14 GB (FP8) |
| **Speed** | ~2x faster | Baseline |
| **Quality** | Great, strong text following | Best overall |
| **VRAM** | ~12 GB peak | ~15 GB peak |
| **Best for** | Quick iterations, text-heavy prompts | Final quality, I2V |

HunyuanVideo is ideal when you need **faster generation** or want to **experiment with prompts** before committing to a longer Wan render.

**Run cells 1-6 in order**, then open the tunnel URL.

In [ ]:
#@title 1. Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Install ComfyUI + Custom Nodes
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Already cloned"
%cd /content/ComfyUI
!pip install -r requirements.txt -q

# Custom nodes
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git 2>/dev/null || echo "Already cloned"
!git clone https://github.com/kijai/ComfyUI-KJNodes.git 2>/dev/null || echo "Already cloned"

!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q
!pip install -r ComfyUI-KJNodes/requirements.txt -q

print("\nInstalled!")

In [ ]:
#@title 3. Download HunyuanVideo Models (~15 GB)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)
os.makedirs(f"{M}/clip", exist_ok=True)

files = {
    # HunyuanVideo transformer FP8 (~7.76 GB)
    f"{M}/diffusion_models/hunyuan_video_v2_replace_fp8_e4m3fn.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/diffusion_models/hunyuan_video_v2_replace_fp8_e4m3fn.safetensors",
    # LLM text encoder FP8 (~4.5 GB)
    f"{M}/text_encoders/llava_llama3_fp8_scaled.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/text_encoders/llava_llama3_fp8_scaled.safetensors",
    # CLIP text encoder (~246 MB)
    f"{M}/clip/clip_l.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/text_encoders/clip_l.safetensors",
    # VAE (~826 MB)
    f"{M}/vae/hunyuan_video_vae_bf16.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/vae/hunyuan_video_vae_bf16.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Already exists: {os.path.basename(dest)}")
    else:
        print(f"\nDownloading {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

print("\nAll models ready!")
!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/clip/* {M}/vae/*

In [ ]:
#@title 4. Download Workflow from Gist
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

!wget -q -O "{WF_DIR}/video_hunyuan_t2v.json" "{RAW}/video_hunyuan_t2v.json"
print("Workflow downloaded!")

In [ ]:
#@title 5. Upload Media (optional — for future I2V support)
from google.colab import files
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(INPUT_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"Saved: {path}")

print(f"\nFiles in input/: {os.listdir(INPUT_DIR)}")

In [ ]:
#@title 6. Launch ComfyUI
import subprocess, time, re

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start ComfyUI
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print("Starting ComfyUI... (30s)")
time.sleep(30)

# Cloudflare tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(5)

url = None
for _ in range(20):
    line = tunnel.stdout.readline().decode()
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print(f"\n{'='*60}")
    print(f"ComfyUI ready! Open: {url}")
    print(f"{'='*60}")
    print("\nLoad workflow: Menu -> Load -> video_hunyuan_t2v")
else:
    print("Tunnel URL not found. ComfyUI running on port 8188.")

In [ ]:
#@title 7. Save Outputs to Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/hunyuan"
os.makedirs(dst, exist_ok=True)

videos = glob.glob(f"{src}/*.mp4") + glob.glob(f"{src}/*.png")
if not videos:
    print("No outputs found yet. Generate some videos first!")
else:
    for v in videos:
        shutil.copy2(v, dst)
        print(f"Copied: {os.path.basename(v)}")
    print(f"\n{len(videos)} files saved to {dst}")

---
## Usage Guide

### video_hunyuan_t2v — Text-to-Video
1. Open the tunnel URL
2. Load **video_hunyuan_t2v** workflow
3. Edit the text prompt
4. Click **Queue Prompt**

### Resolution Presets (16:9)
| Size | Frames | Duration | VRAM |
|------|--------|----------|------|
| 848x480 | 61 | ~4s | ~12 GB |
| 848x480 | 97 | ~6s | ~14 GB |
| 640x368 | 129 | ~8s | ~13 GB |

### Tips
- HunyuanVideo follows text instructions very precisely
- Use detailed, specific prompts for best results
- Keep frame count under 129 on T4 to avoid OOM
- Output is 16fps (vs Wan's 24fps)

### Prompt Style
HunyuanVideo works best with **descriptive, natural language prompts**:
> "A golden retriever running through a sunlit meadow, wildflowers swaying in the breeze, cinematic camera tracking shot, warm afternoon light"